# Lambert Liu Variants

In [ ]:
import utils as ut
import b_run_staging as b
import h_ll_runner as h
from i_valid_and_test_runs import Tuner
import c_clustering as c
import a_preprocessing as a
import n_ll_variants as n
import numpy as np
import polars as pl
from numba import set_num_threads, get_num_threads

set_num_threads(15)
get_num_threads()

15

### Loading in required data and changing to named tuples

In [ ]:
# Loading in required numpy arrays
static_configs = ut.load_json5('static_configs')
runtime_configs = ut.load_json5('runtime_configs')
base_config = ut.merge_configs(static_configs, runtime_configs)

train_test_dict = a.get_train_test_split()
bin_metric_dict = a.add_training_denom(a.get_bin_metrics())
hyperparams = ut.load_json5('hyper_choices')

user_counts = ut.load_data('user_counts', 'df')
user_interactions = ut.load_data('user_interactions', 'df')
user_mapping = ut.load_data('user_mapping', 'df')

degen_mask = ut.load_data('degen_mask', 'np')
interpolation_weights = ut.load_data('interpolation_weights', 'np')
quadratic_interpolation_weights = ut.load_data('quadratic_interpolation_weights', 'np', data_dir=f'{ut.data_dir}/variants')

# Loading initial grids
u_init = ut.load_data('u_init', 'np')
v_init = ut.load_data('v_init', 'np')

p_init = ut.load_data('p_init', 'np')
u_pos_init = ut.load_data('u_pos_init', 'np')
v_pos_init = ut.load_data('v_pos_init', 'np')

n_counts_init = ut.load_data('n_counts_init', 'np')

u_clustering = ut.load_data('u_clustering', 'np')
v_clustering = ut.load_data('v_clustering', 'np')

u_pos_clustering = ut.load_data('u_pos_clustering', 'np')
v_pos_clustering = ut.load_data('v_pos_clustering', 'np')
p_pos_clustering = ut.load_data('p_pos_clustering', 'np')


### Converting to named tuples

In [ ]:
# Converting the dfs to nt of numpy arrays to be used for the final numba runner
user_interactions_nt = b.df_to_nt('user_interactions_nt', user_interactions)
user_counts_nt = b.df_to_nt('user_counts_nt', user_counts,)
output_idx_nt = b.get_model_and_output_idx_nt()
train_test_nt_class = b.dictionary_to_named_tuple_class('train_test_nt',train_test_dict)
train_test_nt = train_test_nt_class(**train_test_dict)
bin_metric_nt = b.dictionary_to_named_tuple_class('bin_metric_nt', bin_metric_dict)(**bin_metric_dict)

### Creating tuner class for runs

In [ ]:
# Creating user type groups for tuning
user_type_groups = (user_mapping.sort('user_id')['source_user_type'].to_numpy() == 'machine')

t = Tuner(u_init, v_init, p_init, u_pos_init, v_pos_init, u_clustering, v_clustering, u_pos_clustering, v_pos_clustering, p_pos_clustering, 
          n_counts_init, user_counts_nt, user_interactions_nt, interpolation_weights, bin_metric_nt, output_idx_nt, train_test_nt_class, user_type_groups)

# Validation Runs

### Weekly runner

In [ ]:
weekly_results, weekly_degen_mask = n.run_weekly_variant(t=t, user_counts=user_counts, user_mapping=user_mapping, train_test_dict=train_test_dict, 
                bin_metric_dict=bin_metric_dict, static_configs=static_configs, base_config=base_config, hyperparams=hyperparams)

finished_config 1/10 in 80.4s
finished_config 2/10 in 61.1s
finished_config 3/10 in 61.3s
finished_config 4/10 in 62.6s
finished_config 5/10 in 61.0s
finished_config 6/10 in 61.4s
finished_config 7/10 in 61.1s
finished_config 8/10 in 60.6s
finished_config 9/10 in 61.7s
finished_config 10/10 in 58.7s


#### Daily runner with weekly mask

In [ ]:
daily_weekly_mask_results = n.run_daily_weekly_mask_variant(t=t, weekly_degen_mask=weekly_degen_mask, hyperparams=hyperparams, 
                                                             train_test_dict=train_test_dict, base_config=base_config)

finished_config 1/10 in 89.3s
finished_config 2/10 in 69.7s
finished_config 3/10 in 70.8s
finished_config 4/10 in 71.0s
finished_config 5/10 in 71.3s
finished_config 6/10 in 73.5s
finished_config 7/10 in 71.4s
finished_config 8/10 in 70.8s
finished_config 9/10 in 71.2s
finished_config 10/10 in 71.1s


#### Quadratic interpolation runner

In [ ]:
quadratic_results = n.run_quadratic_variant(t=t, quadratic_interpolation_weights=quadratic_interpolation_weights, hyperparams=hyperparams, train_test_dict=train_test_dict, base_config=base_config, degen_mask=degen_mask)

finished_config 1/10 in 84.7s
finished_config 2/10 in 68.1s
finished_config 3/10 in 68.7s
finished_config 4/10 in 70.1s
finished_config 5/10 in 69.2s
finished_config 6/10 in 69.9s
finished_config 7/10 in 68.8s
finished_config 8/10 in 69.0s
finished_config 9/10 in 69.7s
finished_config 10/10 in 69.7s


#### NB runner

In [ ]:
nb_results = n.run_nb_variant(t=t, hyperparams=hyperparams, train_test_dict=train_test_dict, base_config=base_config, degen_mask=degen_mask)

finished_config 1/10 in 80.2s
finished_config 2/10 in 64.3s
finished_config 3/10 in 64.9s
finished_config 4/10 in 64.7s
finished_config 5/10 in 65.2s
finished_config 6/10 in 64.5s
finished_config 7/10 in 64.0s
finished_config 8/10 in 64.3s
finished_config 9/10 in 63.9s
finished_config 10/10 in 63.8s


#### Storing variants Results

In [2]:
n.store_variant_results()